# Test process_r.py — Local End-to-End Validation

This notebook tests the `input_transform_intensities` Python runner locally, **without AWS buckets or the MD platform**. It uses in-memory sample data to verify that:

1. The Python runner triggers the R workflow correctly
2. The R function runs and returns the expected output structure
3. Output has the required fields: `intensity`, `metadata`, `runtime_metadata`

**Prerequisites:** Run from the MDCustomR repo root. Install the Python package (`pip install -e .`) and the R package (`Rscript install.R` or `devtools::install()`).

---

## Troubleshooting some common errors

rpy2 (used by `run_r_task`) may use a **different R** than the one in your terminal. Packages installed in one R are invisible to the other.

**Check which R Jupyter is using** (run in a notebook cell):

```python
import rpy2.robjects as ro
print("R_HOME:", ro.r('R.home()')[0])
print("Library paths:", list(ro.r('.libPaths()')))
print("R version:", ro.r('R.version.string')[0])
```

**Compare with your terminal:** run `which R`, `R --version`, and `Rscript -e ".libPaths()"`.

If they differ, install your R package and its dependencies into the R that Jupyter/rpy2 uses. Use the full path to that R's `Rscript` (shown by `R.home()` above) when running `dependencies.R` and `install.R`. If using conda, prefer `conda install r-<pkg> -c conda-forge` for CRAN packages, and `BiocManager::install()` for Bioconductor packages (e.g. limma).

After installing, **restart the Jupyter kernel** before re-running.

For a conda environment, example commands:

```bash
# Install devtools and CRAN dependencies via conda (avoids compilation issues)
conda install r-devtools r-tidyr r-dplyr r-tidyselect -c conda-forge -y

# Install Bioconductor packages (limma) into conda R
/opt/anaconda3/envs/<your_env>/bin/Rscript -e "
if (!require('BiocManager', quietly=TRUE)) install.packages('BiocManager', repos='https://cloud.r-project.org')
BiocManager::install('limma')
"

# Install MDCustomR into conda R
cd /path/to/MDCustomR
/opt/anaconda3/envs/<your_env>/bin/Rscript install.R

# Verify
/opt/anaconda3/envs/<your_env>/bin/Rscript -e "library(MDCustomR); print('OK')"
```

After installing, **restart the Jupyter kernel** before re-running.

In [1]:
import os
import sys

# Set working directory to repo root (parent of tutorial/)
# Run this notebook from MDCustomR root, or from tutorial/ — we adjust accordingly
cwd = os.getcwd()
repo_root = os.path.dirname(cwd) if os.path.basename(cwd) == "tutorial" else cwd
os.chdir(repo_root)
sys.path.insert(0, repo_root)

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/mansiaggarwal/Desktop/Main/Git_Repos/MDCustomR


## 1. Create sample intensity and metadata dataframes

The `transformIntensities` R function expects:
- **Intensity:** `GroupId`, `replicate`, `NormalisedIntensity` (plus optional columns)
- **Metadata:** `GeneNames`, `GroupId`, `GroupLabel`, `GroupLabelType`, `ProteinIds`, `Description`

In [2]:
import pandas as pd
import numpy as np

# Minimal intensity data: 3 proteins x 3 replicates
# NormalisedIntensity: raw values (R will apply log2 before normalization)
intensity_data = pd.DataFrame({
    "GroupId": [1, 1, 1, 2, 2, 2, 3, 3, 3],
    "replicate": ["rep1", "rep2", "rep3"] * 3,
    "NormalisedIntensity": [100.0, 105.0, 98.0, 200.0, 210.0, 195.0, 50.0, 52.0, 48.0],
})

# Metadata for the same GroupIds
metadata_data = pd.DataFrame({
    "GroupId": [1, 2, 3],
    "GeneNames": ["GeneA", "GeneB", "GeneC"],
    "GroupLabel": ["Protein A", "Protein B", "Protein C"],
    "GroupLabelType": ["Protein", "Protein", "Protein"],
    "ProteinIds": ["P1", "P2", "P3"],
    "Description": ["", "", ""],
})

print("Intensity shape:", intensity_data.shape)
print("Metadata shape:", metadata_data.shape)
intensity_data

Intensity shape: (9, 3)
Metadata shape: (3, 6)


,GroupId,replicate,NormalisedIntensity
0,1,rep1,100.0
1,1,rep2,105.0
2,1,rep3,98.0
3,2,rep1,200.0
4,2,rep2,210.0
5,2,rep3,195.0
6,3,rep1,50.0
7,3,rep2,52.0
8,3,rep3,48.0


In [3]:
import uuid
from md_dataset.models.dataset import (
    InputDatasetTable,
    IntensityInputDataset,
    IntensityTableType,
    DatasetType,
)

# Table names must match what IntensityInputDataset.table() expects
# ("Protein_Intensity" and "Protein_Metadata")
intensity_table = InputDatasetTable(name="Protein_Intensity", data=intensity_data)
metadata_table = InputDatasetTable(name="Protein_Metadata", data=metadata_data)

input_dataset = IntensityInputDataset(
    id=uuid.uuid4(),
    name="Test Intensity Dataset",
    type=DatasetType.INTENSITY,
    tables=[intensity_table, metadata_table],
)

print(f"Input dataset: {input_dataset.name}")
print(f"Tables: {[t.name for t in input_dataset.tables]}")

Input dataset: Test Intensity Dataset
Tables: ['Protein_Intensity', 'Protein_Metadata']


## 3. Call the Python runner and execute R

The `input_transform_intensities` function returns `RFuncArgs` (arguments for R), not the R output. The `@md_r` decorator only runs R when the function is invoked inside a Prefect flow on the MD platform. Hence, to run the following code chunk, temporarily comment the lines with the decorate (e.g., line 25 and 42 in `src/md_custom_r/process_r.py`).

In [4]:
from src.md_custom_r.process_r import input_transform_intensities, MDCustomRParams
from md_dataset.process import run_r_task

params = MDCustomRParams(normalisation_methods="scale")

# Step 1: Get RFuncArgs (arguments that would be passed to R on the platform)
r_func_args = input_transform_intensities(
    input_datasets=[input_dataset],
    params=params,
    output_dataset_type=DatasetType.INTENSITY,
)

# Step 2: Execute R locally to get actual workflow output
output = run_r_task(
    r_file="./src/md_custom_r/process.R",
    r_function="run_transform_intensities",
    r_preparation=r_func_args,
)

/opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/pydantic_settings/main.py:447: UserWarning: Config key `pyproject_toml_table_header` is set in model_config but will be ignored because no PyprojectTomlConfigSettingsSource source is configured. To use this config key, add a PyprojectTomlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  cls._settings_warn_unused_config_keys(sources, cls.model_config)
/opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/pydantic_settings/main.py:447: UserWarning: Config key `toml_file` is set in model_config but will be ignored because no TomlConfigSettingsSource source is configured. To use this config key, add a TomlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  cls._settings_warn_unused_config_keys(sources, cls.model_config)


11:31:53.420 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8634
See https://docs.prefect.io/3.0/manage/self-host#self-host-a-prefect-server for more information on running a dedicated Prefect server.

/opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/pydantic_settings/main.py:447: UserWarning: Config key `pyproject_toml_table_header` is set in model_config but will be ignored because no PyprojectTomlConfigSettingsSource source is configured. To use this config key, add a PyprojectTomlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  cls._settings_warn_unused_config_keys(sources, cls.model_config)
/opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/pydantic_settings/main.py:447: UserWarning: Config key `toml_file` is set in model_config but will be ignored because no TomlConfigSettingsSource source is configured. To use this config key, add a TomlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  cls._settings_warn_unused_config_keys(sources, cls.model_config)


11:31:56.572 | WARNING | rpy2.rinterface_lib.openrlib - Error importing in API mode: ImportError("dlopen(/opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <587B6A46-0BED-3992-853A-486017AC0683> /opt/anaconda3/envs/md_custom_r/lib/python3.11/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")

11:31:56.574 | WARNING | rpy2.rinterface_lib.openrlib - Trying to import in ABI mode.

...

11:31:57.704 | INFO    | Task run 'run_r_task' - Convert named ListVector to dict

11:31:57.707 | INFO    | Task run 'run_r_task' - Convert: <class 'pandas.core.frame.DataFrame'>

11:31:57.709 | INFO    | Task run 'run_r_task' - Convert: <class 'pandas.core.frame.DataFrame'>

11:31:57.710 | INFO    | Task run 'run_r_task' - Convert: <class 'pandas.core.frame.DataFrame'>

11:31:57.712 | INFO    | Task run 'run_r_task' - Finished in state Completed()

> ⚠️ **Important:** Make sure to uncomment the `@md_r` decorator lines (e.g., line 25 and 42 in `src/md_custom_r/process_r.py`) before pushing to the platform.

## 4. Verify RFuncArgs input and run R

First verify the Python runner produced correct `RFuncArgs` (data passed to R). `r_func_args.data_frames[0]` = Protein_Intensity, `r_func_args.data_frames[1]` = metadata. Then run R via `run_r_task` and verify the output.

In [5]:
# Verify RFuncArgs: data_frames[0] = Protein_Intensity, data_frames[1] = metadata
assert len(r_func_args.data_frames) == 2, "RFuncArgs should have 2 data_frames"

print("r_func_args.data_frames[0] is Protein_Intensity (intensity input):")
display(r_func_args.data_frames[0].head())

print("r_func_args.data_frames[1] is metadata:")
display(r_func_args.data_frames[1].head())

r_func_args.data_frames[0] is Protein_Intensity (intensity input):


,GroupId,replicate,NormalisedIntensity
0,1,rep1,100.0
1,1,rep2,105.0
2,1,rep3,98.0
3,2,rep1,200.0
4,2,rep2,210.0


r_func_args.data_frames[1] is metadata:


,GroupId,GeneNames,GroupLabel,GroupLabelType,ProteinIds,Description
0,1,GeneA,Protein A,Protein,P1,
1,2,GeneB,Protein B,Protein,P2,
2,3,GeneC,Protein C,Protein,P3,


In [6]:
# Verify R output has required fields (output comes from run_r_task)
required = ["intensity", "metadata"]
optional = ["runtime_metadata"]

assert "intensity" in output, "Missing required 'intensity' in output"
assert "metadata" in output, "Missing required 'metadata' in output"

print("✓ Output has required fields: intensity, metadata")

if "runtime_metadata" in output:
    print("✓ Output has runtime_metadata")
    print("Runtime metadata:")
    display(output["runtime_metadata"])

print("\nIntensity output shape:", output["intensity"].shape)
print("Metadata output shape:", output["metadata"].shape)

✓ Output has required fields: intensity, metadata
✓ Output has runtime_metadata
Runtime metadata:


,RVersion,replicateColname,featureColname,normMethod
1,R version 4.5.2 (2025-10-31),replicate,GroupId,scale



Intensity output shape: (9, 3)
Metadata output shape: (3, 6)


In [7]:
# Show transformed intensity (NormalisedIntensity updated by R workflow)
output["intensity"]

,GroupId,replicate,NormalisedIntensity
1,1.0,rep1,6.657473
2,1.0,rep2,6.657473
3,1.0,rep3,6.657473
4,2.0,rep1,7.659523
5,2.0,rep2,7.649017
6,2.0,rep3,7.656511
7,3.0,rep1,5.655423
8,3.0,rep2,5.652239
9,3.0,rep3,5.621068
